# MetroCrowdManager — GRPO on Hugging Face Jobs (A100)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiwakar1997/gluon_openenv/blob/main/notebooks/train_on_hf_jobs.ipynb)

**This notebook is the official re-runnable training entry point** for the
MetroCrowdManager OpenEnv hackathon submission.

It does **not** train the model on the Colab runtime (Colab T4 cannot fit
Gemma-3-27B + GRPO). Instead it submits a real GRPO training job to
**Hugging Face Jobs A100-80GB**, tails the live logs back into the cell
output, embeds the live [Trackio dashboard](https://huggingface.co/spaces/DhiwakarDev/mcm-trackio),
and at end-of-run downloads `rewards.csv` from the Hub repo and renders
loss + reward curves inline.

Pipeline:

```
Colab notebook  ──hf jobs run──▶  HF Jobs A100  ──HTTP──▶  OpenEnv Space
       │                                │                    (rewards)
       │                                │
       │       fetch_job_logs(follow)   │
       │ ◀──────────────────────────────┘
       │
       │       hf_hub_download(rewards.csv, log_history.json)
       │ ◀──── after job ends
       │
       └─▶  matplotlib plots → docs/plots/*.png
```

**Requirements to actually run the training:**
1. A Hugging Face account with [Jobs billing enabled](https://huggingface.co/docs/huggingface_hub/guides/jobs)
   (A100-large is paid).
2. An `HF_TOKEN` with `write` scope, configured in the **Auth** cell below.
3. The OpenEnv Space (`DhiwakarDev/openenv` by default) up and running.

Judges who don't want to spin up a paid run can simply scroll through the
saved cell outputs and click the embedded Trackio dashboard.

## 1. Setup

In [ ]:
%pip install -q --upgrade 'huggingface_hub>=0.27' matplotlib pandas trackio

In [ ]:
import os
import time
from pathlib import Path

from huggingface_hub import (
    HfApi,
    fetch_job_logs,
    hf_hub_download,
    inspect_job,
    notebook_login,
    run_job,
)
from IPython.display import IFrame, display, Markdown

## 2. Authenticate with Hugging Face

You need an `HF_TOKEN` with **write** scope. The token must be on an
account that has [Jobs billing enabled](https://huggingface.co/docs/huggingface_hub/guides/jobs).

In [ ]:
notebook_login(write_permission=True)

## 3. Run parameters

Mirrors `scripts/full_run_hf_job.sh`. Edit these to scale the run.

| Knob | What it does | Cost note |
|---|---|---|
| `PHASE` | Which task to train (`ticket_booking` / `ticket_issuance` / `crowd_announcement`) | — |
| `STEPS` | GRPO update steps (`max_steps`). Reward curve is usually informative by ~200. | A100 hours scale linearly |
| `FLAVOR` | HF Jobs hardware. `a100-large` = 1× A100-80GB. | `a100x4` for 4× A100 |
| `TIMEOUT` | Hard kill after this duration | use `"6h"` for safety |

In [ ]:
PHASE              = "ticket_issuance"   # ticket_booking | ticket_issuance | crowd_announcement
MODEL              = "google/gemma-3-27b-it"
STEPS              = 300
FLAVOR             = "a100-large"
TIMEOUT            = "6h"

BATCH_SIZE         = 1
GRAD_ACCUM         = 8
NUM_GENERATIONS    = 2
MAX_COMPLETION_LEN = 512
MAX_SEQ_LEN        = 4096
LR                 = 5e-6
SAVE_STEPS         = 100
DEBUG_MODE         = 0

REPO_URL           = "https://github.com/Dhiwakar1997/gluon_openenv"
ENV_BASE_URL       = "https://dhiwakardev-openenv.hf.space"
TRACKIO_SPACE_ID   = "DhiwakarDev/mcm-trackio"
TRACKIO_PROJECT    = "mcm-gemma3-27b-full"
PUSH_TO_HUB_ID     = f"DhiwakarDev/mcm-gemma3-27b-grpo-{PHASE}"

DOCKER_IMAGE       = "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel"

print(f"phase={PHASE} model={MODEL} steps={STEPS} flavor={FLAVOR}")
print(f"will push adapters + metrics to: {PUSH_TO_HUB_ID}")

## 4. Build the in-container training command

This is a one-liner shell payload that the HF Job container runs. It:
1. Installs deps,
2. Clones this repo,
3. Runs `training/hf_jobs_train_grpo.py` with all the parameters above.

In [ ]:
PIP_DEPS = (
    "'transformers>=4.55,<4.60' 'trl>=1.2,<1.4' 'peft>=0.13,<0.16' "
    "'accelerate>=1.0,<1.5' bitsandbytes datasets trackio 'openenv-core[core]' "
    "hf_transfer pydantic websockets httpx"
)

TRAIN_CMD = (
    f"python training/hf_jobs_train_grpo.py "
    f"--phase {PHASE} "
    f"--model {MODEL} "
    f"--num-episodes {STEPS} "
    f"--batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} --num-generations {NUM_GENERATIONS} "
    f"--max-completion-len {MAX_COMPLETION_LEN} --max-seq-len {MAX_SEQ_LEN} "
    f"--lr {LR} "
    f"--save-steps {SAVE_STEPS} "
    f"--debug-mode {DEBUG_MODE} "
    f"--env-base-url $ENV_BASE_URL "
    f"--trackio-space-id $TRACKIO_SPACE_ID "
    f"--trackio-project {TRACKIO_PROJECT} "
    f"--push-to-hub-id {PUSH_TO_HUB_ID} "
    f"--output-dir outputs/full-{PHASE}"
)

PAYLOAD = (
    "set -e && "
    "apt-get update -qq && apt-get install -y -qq --no-install-recommends git && "
    f"pip install -q {PIP_DEPS} && "
    f"git clone {REPO_URL} /workspace/Gluon && "
    "cd /workspace/Gluon && "
    f"{TRAIN_CMD}"
)

print(PAYLOAD[:600], "...")

## 5. Submit the job to HF Jobs A100

In [ ]:
job = run_job(
    image=DOCKER_IMAGE,
    command=["bash", "-lc", PAYLOAD],
    flavor=FLAVOR,
    timeout=TIMEOUT,
    env={
        "ENV_BASE_URL": ENV_BASE_URL,
        "TRACKIO_SPACE_ID": TRACKIO_SPACE_ID,
        "TRL_EXPERIMENTAL_SILENCE": "1",
    },
    secrets={"HF_TOKEN": os.environ.get("HF_TOKEN", "")},
)
JOB_ID = job.id
print(f"submitted job_id={JOB_ID}")
print(f"track at: https://huggingface.co/jobs/{JOB_ID}")

## 6. Tail the live logs into this cell

Streams stdout/stderr from the running container until the job finishes.
The training script highlights `>>> reward=... <<<` and `>>> loss=... <<<`
lines so you can spot the curve climbing in real time.

In [ ]:
for line in fetch_job_logs(job_id=JOB_ID, follow=True):
    print(line, flush=True)

## 7. Live metrics — Trackio dashboard (embedded)

TRL's `TrackioCallback` streams every step's loss, reward, and per-component
reward breakdown to a public Hugging Face Space. The cell below embeds that
dashboard inline. While the job is running, the curves update live.

In [ ]:
TRACKIO_URL = f"https://{TRACKIO_SPACE_ID.replace('/', '-').lower()}.hf.space"
display(Markdown(f"**Live trackio dashboard:** [{TRACKIO_URL}]({TRACKIO_URL})"))
IFrame(TRACKIO_URL, width=1100, height=720)

## 8. After the job ends — download metrics + render plots

The patched `training/hf_jobs_train_grpo.py` uploads three artifacts to
`PUSH_TO_HUB_ID` at end-of-run:

- the LoRA adapters,
- `metrics/<phase>/rewards.csv`     — every reward score, one row per completion,
- `metrics/<phase>/log_history.json` — TRL trainer log (loss, kl, etc., per step).

We download them here and render the headline plots.

In [ ]:
info = inspect_job(job_id=JOB_ID)
print(f"job status: {info.status if hasattr(info, 'status') else info}")

In [ ]:
rewards_csv = hf_hub_download(
    repo_id=PUSH_TO_HUB_ID,
    filename=f"metrics/{PHASE}/rewards.csv",
    repo_type="model",
)
log_history_json = hf_hub_download(
    repo_id=PUSH_TO_HUB_ID,
    filename=f"metrics/{PHASE}/log_history.json",
    repo_type="model",
)
print(f"rewards.csv      -> {rewards_csv}")
print(f"log_history.json -> {log_history_json}")

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

PLOTS_DIR = Path("docs/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(rewards_csv)
with open(log_history_json) as f:
    log_history = json.load(f)

print(f"rewards rows: {len(df)}    log_history entries: {len(log_history)}")
df.head()

### Reward curve (smoothed)

In [ ]:
def rolling(series, window):
    return series.rolling(window=window, min_periods=1).mean()

# Aggregate to one mean reward per (step, task) BEFORE smoothing — without
# this, multi-completion runs and the early all-zero steps drag the line flat.
per_step = (
    df.groupby(["task", "step"], as_index=False)["reward"]
      .mean()
      .sort_values(["task", "step"])
)

fig, ax = plt.subplots(figsize=(11, 5))
for task, sub in per_step.groupby("task"):
    sub = sub.sort_values("step")
    window = max(1, len(sub) // 5)  # tighter window for small step counts
    ax.plot(
        sub["step"],
        rolling(sub["reward"], window),
        label=task,
        linewidth=2.2,
        marker="o",
        markersize=4,
    )
ax.set_xlabel("Training step")
ax.set_ylabel("Mean episode reward (smoothed)")
ax.set_title(f"GRPO reward curve — {PHASE} ({MODEL})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "reward_curve.png", dpi=140, bbox_inches="tight")
plt.show()

### Training loss curve

In [ ]:
loss_records = [r for r in log_history if "loss" in r and "step" in r]
if loss_records:
    steps = [r["step"] for r in loss_records]
    losses = [r["loss"] for r in loss_records]
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(steps, losses, color="tab:red", linewidth=2)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Loss")
    ax.set_title(f"GRPO training loss — {PHASE} ({MODEL})")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "loss_curve.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("No 'loss' records in log_history — TRL may have logged with a different key.")

### Per-component reward breakdown

The training script logs every reward dimension's contribution per step
into the `breakdown_json` column. Plotting them shows _why_ the headline
reward changed (e.g. `tool_sequence` climbing, `tool_economy` flat).

In [ ]:
breakdown_rows = []
for _, row in df.iterrows():
    try:
        b = json.loads(row["breakdown_json"])
    except Exception:
        continue
    b["step"] = row["step"]
    breakdown_rows.append(b)

if breakdown_rows:
    bdf = pd.DataFrame(breakdown_rows).fillna(0.0)
    components = [c for c in bdf.columns if c != "step"]
    rolled = bdf.groupby("step")[components].mean().reset_index().sort_values("step")
    window = max(1, len(rolled) // 5)  # tighter smoothing for short runs
    fig, ax = plt.subplots(figsize=(11, 6))
    for c in components:
        ax.plot(rolled["step"], rolling(rolled[c], window), label=c, linewidth=1.5)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Reward component (smoothed)")
    ax.set_title(f"Per-component reward — {PHASE}")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "reward_breakdown.png", dpi=140, bbox_inches="tight")
    plt.show()
else:
    print("No parsable breakdown_json rows.")

## 9. (Optional) Cancel a runaway job

In [ ]:
# Uncomment to cancel:
# from huggingface_hub import cancel_job
# cancel_job(job_id=JOB_ID)

## Done

You should now have three PNGs in `docs/plots/`, **rendered exclusively from
the `rewards.csv` and `log_history.json` produced by the HF Job that just
finished above** — there is no fallback to older runs or to local CSV files.

The README references these PNG paths directly, so:

1. Commit `docs/plots/*.png` to surface the plots on the GitHub README.
2. Re-run `scripts/push_env_to_hf_space.sh` (with `FORCE_README=1`) to refresh
   the HF Space landing page with the new numbers + plots.

Each fresh run of this notebook overwrites the same three filenames, so
re-running end-to-end is the canonical way to update the headline plots.

- **Live metrics**: <https://huggingface.co/spaces/DhiwakarDev/mcm-trackio>
- **Adapters + metrics**: <https://huggingface.co/DhiwakarDev>
- **Env Space**: <https://huggingface.co/spaces/DhiwakarDev/openenv>